In [0]:
spark.sql("USE CATALOG relp")
spark.sql("USE SCHEMA bronze")

In [0]:
base = "dbfs:/Volumes/relp/bronze/raw/"
from pyspark.sql.functions import current_timestamp, lit

In [0]:
transactions = spark.read.option("header", True).option("inferSchema", True).csv(base + "transactions.csv")

transactions = transactions.withColumn("ingestion_time", current_timestamp()) \
                           .withColumn("source_file", lit("transactions.csv")) \
                           .withColumn("layer", lit("bronze"))

transactions.write.format("delta").mode("overwrite")


In [0]:
customers = spark.read.option("header", True).option("inferSchema", True).csv(base + "customers.csv")

customers = customers.withColumn("ingestion_time", current_timestamp()) \
                     .withColumn("source_file", lit("customers.csv")) \
                     .withColumn("layer", lit("bronze"))

customers.write.format("delta").mode("overwrite")


In [0]:
properties = spark.read.option("header", True).option("inferSchema", True).csv(base + "properties.csv")

properties = properties.withColumn("ingestion_time", current_timestamp()) \
                       .withColumn("source_file", lit("properties.csv")) \
                       .withColumn("layer", lit("bronze"))

properties.write.format("delta").mode("overwrite")

In [0]:
listings = spark.read.option("header", True).option("inferSchema", True).csv(base + "listings.csv")

listings = listings.withColumn("ingestion_time", current_timestamp()) \
                   .withColumn("source_file", lit("listings.csv")) \
                   .withColumn("layer", lit("bronze"))

listings.write.format("delta").mode("overwrite")


In [0]:
agents = spark.read.option("header", True).option("inferSchema", True).csv(base + "agents.csv")

agents = agents.withColumn("ingestion_time", current_timestamp()) \
               .withColumn("source_file", lit("agents.csv")) \
               .withColumn("layer", lit("bronze"))

agents.write.format("delta").mode("overwrite")

In [0]:
interactions = spark.read.option("header", True).option("inferSchema", True).csv(base + "interactions.csv")

interactions = interactions.withColumn("ingestion_time", current_timestamp()) \
                           .withColumn("source_file", lit("interactions.csv")) \
                           .withColumn("layer", lit("bronze"))

interactions.write.format("delta").mode("overwrite")


In [0]:
customer_monthly_metrics = spark.read.option("header", True).option("inferSchema", True).csv(base + "customer_monthly_metrics.csv")

customer_monthly_metrics = customer_monthly_metrics.withColumn("ingestion_time", current_timestamp()) \
                                                   .withColumn("source_file", lit("customer_monthly_metrics.csv")) \
                                                   .withColumn("layer", lit("bronze"))

customer_monthly_metrics.write.format("delta").mode("overwrite")


In [0]:
import importlib
import pipeline.bronze.ingestion as ingestion

In [0]:
importlib.reload(ingestion)
from pipeline.bronze.ingestion import process_bronze_table

base = "dbfs:/Volumes/relp/bronze/raw/"

tables = {
    "transactions": "transaction_id",
    "customers": "customer_id",
    "properties": "property_id",
    "listings": "listing_id",
    "agents": "agent_id",
    "interactions": "interaction_id",
    "customer_monthly_metrics": None
}

results = {}

print("=" * 60)
print("🚀 BRONZE LAYER STARTED")
print("=" * 60)

for table, pk in tables.items():
    df = process_bronze_table(spark, base, table, pk)
    results[table] = df.count()

print("\n" + "=" * 60)
print("📊 FINAL SUMMARY")
print("=" * 60)

for table, count in results.items():
    print(f"{table:<30} → {count} rows")

print("\n📂 Tables in Catalog:")
display(spark.sql("SHOW TABLES"))

In [0]:
spark.sql("SELECT ingestion_time, source_file, layer FROM bronze_transactions LIMIT 5").show()